In [2]:
import cv2
import os
import pickle
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet

# 1. Khởi tạo mô hình
detector = MTCNN()
embedder = FaceNet()
BASE_PATH = 'face_image_set'
DB_FILE = "face_memory.pkl"

known_embeddings = []
known_names = []

# Kiểm tra nếu file đã tồn tại thì nạp luôn, nếu không mới quét folder
if os.path.exists(DB_FILE):
    print(f"--- Đang nạp bộ nhớ từ file {DB_FILE} ---")
    with open(DB_FILE, "rb") as f:
        data = pickle.load(f)
        known_embeddings = data["embeddings"]
        known_names = data["names"]
    print(f"Hoàn tất! Đã nạp {len(known_names)} khuôn mặt.")
else:
    print("--- Không thấy file bộ nhớ. Bắt đầu quét thư mục ảnh mẫu ---")
    for person_name in os.listdir(BASE_PATH):
        person_dir = os.path.join(BASE_PATH, person_name)
        if os.path.isdir(person_dir):
            print(f"Đang xử lý: {person_name}")
            for img_name in os.listdir(person_dir):
                img_path = os.path.join(person_dir, img_name)
                img = cv2.imread(img_path)
                if img is None: continue

                # Resize chống MemoryError
                h, w = img.shape[:2]
                if max(h, w) > 640:
                    scale = 640 / max(h, w)
                    img = cv2.resize(img, (int(w * scale), int(h * scale)))

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                res = detector.detect_faces(img_rgb)
                if res:
                    x, y, w, h = res[0]['box']
                    x, y = max(0, x), max(0, y)
                    face = img_rgb[y:y+h, x:x+w]
                    
                    if face.size > 0:
                        face_resized = cv2.resize(face, (160, 160))
                        face_exp = np.expand_dims(face_resized, axis=0)
                        embedding = embedder.embeddings(face_exp)[0]
                        
                        known_embeddings.append(embedding)
                        known_names.append(person_name)
                        print(f"  v Thành công: {img_name}")
    
    # LƯU FILE SAU KHI QUÉT XONG
    if known_embeddings:
        with open(DB_FILE, "wb") as f:
            pickle.dump({"embeddings": known_embeddings, "names": known_names}, f)
        print(f"\n--- ĐÃ LƯU FILE {DB_FILE} THÀNH CÔNG ---")

--- Không thấy file bộ nhớ. Bắt đầu quét thư mục ảnh mẫu ---
Đang xử lý: Henry Cavill
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
  v Thành công: Henry Cavill_0.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
  v Thành công: Henry Cavill_1.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
  v Thành công: Henry Cavill_10.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
  v Thành công: Henry Cavill_100.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
  v Thành công: Henry Cavill_101.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
  v Thành công: Henry Cavill_102.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
  v Thành công: Henry Cavill_103.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
  v Thành công: Henry Cavill_104.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
  v Thành công: Henry Cavill_105.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
  v Thành công: Henry Cavill_11.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
  v Thành công: Henry Cavill_12.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
  v Thành công: Henry Cavill_13.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━

In [3]:
THRESHOLD = 0.7  # Ngưỡng nhận diện

def get_embedding(face_img):
    face_img = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    face_img = cv2.resize(face_img, (160, 160))
    face_img = np.expand_dims(face_img, axis=0)
    embedding = embedder.embeddings(face_img)
    return embedding[0]

print("\n--- Đang mở Webcam... Nhấn 'Q' tại cửa sổ ảnh để dừng ---")
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break

    results = detector.detect_faces(frame)
    
    for res in results:
        x, y, w, h = res['box']
        x, y = max(0, x), max(0, y)
        face_img_crop = frame[y:y+h, x:x+w]
        
        if face_img_crop.size > 0:
            # Trích xuất đặc trưng mặt hiện tại
            curr_embedding = get_embedding(face_img_crop)
            max_sim = -1
            best_match = "Unknown"
            
            # So sánh với bộ nhớ đã nạp từ file .pkl
            if len(known_embeddings) > 0:
                for i, known_emb in enumerate(known_embeddings):
                    # Tính Cosine Similarity
                    sim = np.dot(curr_embedding, known_emb) / (np.linalg.norm(curr_embedding) * np.linalg.norm(known_emb))
                    if sim > max_sim:
                        max_sim = sim
                        best_match = known_names[i]
            
            # Hiển thị
            label = f"{best_match}" if max_sim > THRESHOLD else "Unknown"
            color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
            
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
            cv2.putText(frame, f"{label}: {max_sim:.2f}", (x, y-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imshow('Face Recognition System', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


--- Đang mở Webcam... Nhấn 'Q' tại cửa sổ ảnh để dừng ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 6